# Analisis de resultados: OWSM adaptado a senal neuronal

Este cuaderno genera las figuras y tablas de la memoria a partir de los artefactos descargados de los entrenamientos. No entrena modelos.

El analisis mantiene separadas tres preguntas:

1. **Transferencia:** si los pesos acusticos preentrenados de OWSM ayudan al pasar a sEEG.
2. **Capacidad:** si la arquitectura E-Branchformer puede aprender la tarea desde inicializacion aleatoria.
3. **Sistema final:** que decisiones reducen el error de palabra (WER), que es la metrica principal de uso.

El CER se conserva para estudiar el encoder y las curvas de entrenamiento. Al comparar unidades CTC distintas solo se utilizan CER/WER calculados sobre el **texto reconstruido**; el error CTC nativo de caracteres, fonemas, palabras y BPE no es comparable entre objetivos.

## Orden experimental unificado

| Orden | Bloque | Experimento |
|---:|:---:|---|
| 1 | A | Transferencia del encoder |
| 2 | C | Estabilidad del entrenamiento |
| 3 | D | Estrategia de adaptación |
| 4 | E | Frontend de EEG |
| 5 | F | Profundidad del encoder |
| 6 | G | Unidad de salida CTC |
| 7 | H | Generalización entre sesiones |
| 8 | I | Modelo de lenguaje externo |
| 9 | J | Decoder OWSM preliminar |
| 10 | B | Sistema final |

La letra `P` identifica pruebas preliminares conservadas únicamente por trazabilidad. Los nombres históricos (`v3`, `B3word`, etc.) siguen siendo las claves físicas de los artefactos, pero las figuras y tablas usan los identificadores científicos anteriores.



In [ ]:
from pathlib import Path
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display, Markdown

# Se puede fijar TFG_RESULTS_DIR antes de ejecutar el notebook.
RESULTS_OVERRIDE = os.environ.get("TFG_RESULTS_DIR")
OUTPUT_OVERRIDE = os.environ.get("TFG_ANALYSIS_OUTPUT_DIR")
EXPORTAR = True
DPI = 300

if Path("/content").exists() and not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass

candidatos = []
if RESULTS_OVERRIDE:
    candidatos.append(Path(RESULTS_OVERRIDE))
candidatos += [
    Path("../resultados"),
    Path.cwd() / "../resultados",
    Path.cwd() / "resultados",
    Path("/content/drive/MyDrive/TFG/resultados"),
]

RESULTS_DIR = next(
    (p.resolve() for p in candidatos if (p / "resultados.csv").exists()),
    None,
)
if RESULTS_DIR is None:
    buscados = "\n".join(f"  - {p}" for p in candidatos)
    raise FileNotFoundError(f"No se encontro resultados.csv. Rutas comprobadas:\n{buscados}")

OUTPUT_ROOT = Path(OUTPUT_OVERRIDE).resolve() if OUTPUT_OVERRIDE else RESULTS_DIR
FIG_DIR = OUTPUT_ROOT / "figuras_memoria"
TABLE_DIR = OUTPUT_ROOT / "tablas_memoria"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

COLORS = {
    "random": "#D55E00",
    "owsm": "#0072B2",
    "neutral": "#4D4D4D",
    "green": "#009E73",
    "yellow": "#E69F00",
    "red": "#C44E52",
    "purple": "#7A5195",
    "light": "#B8B8B8",
}

plt.rcParams.update({
    "figure.figsize": (7.2, 4.4),
    "figure.dpi": 120,
    "savefig.dpi": DPI,
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "legend.frameon": False,
    "legend.fontsize": 9,
    "lines.linewidth": 2.2,
    "lines.markersize": 6,
})

def guardar(fig, nombre):
    if EXPORTAR:
        for ext in ("png", "pdf"):
            fig.savefig(FIG_DIR / f"{nombre}.{ext}", bbox_inches="tight", dpi=DPI)

def exportar_tabla(tabla, nombre, index=False):
    tabla.to_csv(TABLE_DIR / f"{nombre}.csv", index=index)
    try:
        texto = tabla.to_latex(index=index, float_format=lambda x: f"{x:.3f}")
        (TABLE_DIR / f"{nombre}.tex").write_text(texto, encoding="utf-8")
    except Exception as exc:
        warnings.warn(f"No se pudo exportar {nombre}.tex: {exc}")

def eje_porcentaje(ax, axis="x", xmax=None):
    target = ax.xaxis if axis == "x" else ax.yaxis
    target.set_major_formatter(PercentFormatter(1.0, decimals=0))
    if xmax is not None:
        (ax.set_xlim if axis == "x" else ax.set_ylim)(0, xmax)

def anotar_barras_h(ax, valores, offset=0.01):
    for patch, valor in zip(ax.patches, valores):
        if pd.notna(valor):
            ax.text(valor + offset, patch.get_y() + patch.get_height() / 2,
                    f"{valor:.1%}", va="center", fontsize=9)

print("Resultados:", RESULTS_DIR)
print("Figuras:   ", FIG_DIR)
print("Tablas:    ", TABLE_DIR)


: 

## 1. Carga, limpieza y trazabilidad

Los experimentos antiguos no guardaban todos los metadatos que incorporaron las versiones posteriores. Solo se reconstruyen campos demostrables por el nombre del experimento y por el codigo historico: aquellas versiones siempre cargaban OWSM, empleaban BPE-50k, normalizacion por ensayo, submuestreo por cuatro y no implementaban InterCTC. La columna `metadatos_inferidos` permite identificarlos.


In [ ]:
df = pd.read_csv(RESULTS_DIR / "resultados.csv")
df.columns = [c.strip() for c in df.columns]

numeric_cols = [
    "n_train", "n_val", "n_sessions", "max_epoch", "epocas_hechas", "seed",
    "ctc_weight", "enc_lr_scale", "lr", "weight_decay", "grad_clip",
    "subsample", "interctc_weight", "n_capas_encoder", "unfreeze_ultimas",
    "holdout_sesiones", "compute_units", "minutos",
    "min_valid_cer_ctc", "best_valid_cer_ctc", "epoch_min_cer_ctc",
    "cer_ctc_micro", "cer_ctc_greedy", "cer_ctc_greedy_se", "wer_ctc_greedy",
    "cer_beam", "wer_beam", "best_valid_cer", "best_valid_wer",
]
for col in numeric_cols:
    if col in df:
        df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ("error", "objetivo_ctc", "init_encoder", "frontend", "exp_tag", "nombre"):
    if col not in df:
        df[col] = ""

# Artefactos previos a la introduccion explicita de init_encoder.
legacy_owsm_tags = {
    "sanity_n2_linear_unf1_lora1_lr0.001_els1_ctc0.3_bs16_ep15",
    "base_n10_conv2d_unf1_lora1_lr0.001_els1_ctc0.3_bs16_ep30",
    "lrdif_n10_conv2d_unf1_lora1_lr0.001_els0.01_ctc0.3_bs16_ep30",
    "linear_n10_linear_unf1_lora1_lr0.001_els1_ctc0.3_bs16_ep30",
    "conv1d_n10_conv1d_unf1_lora1_lr0.001_els1_ctc0.3_bs16_ep30",
    "frozen_n10_conv2d_unf0_lora1_lr0.001_els1_ctc0.3_bs16_ep30",
    "ctc00_n10_conv2d_unf1_lora1_lr0.001_els1_ctc0_bs16_ep30",
    "ctc05_n10_conv2d_unf1_lora1_lr0.001_els1_ctc0.5_bs16_ep30",
    "overfit_n1_linear_unf1_lora1_lr0.001_els1_ctc1_bs16_ep200",
    "sinSesion_n10_linear_unf1_lora1_lr0.001_els0.01_ctc0.3_bs16_ep30",
    "conSesion_n10_linear_sess_unf1_lora1_lr0.001_els0.01_ctc0.3_bs16_ep30",
    "gen1_n1_linear_unf1_lora1_lr0.001_els1_ctc1_bs16_ep200",
    "charOverfit_n1_linear_char_unf1_lora1_lr0.001_els1_ctc1_bs16_ep200",
    "charGen1_n1_linear_char_unf1_lora1_lr0.001_els1_ctc1_bs16_ep200",
    "charOverfit_n1_linear_char_unf1_lora1_lr0.001_els1_ctc1_bs16_ep50",
    "charGen1_n1_linear_char_unf1_lora1_lr0.001_els1_ctc1_bs16_ep150",
    "fullCon_n45_linear_sess_char_unf1_lora1_lr0.001_els1_ctc1_bs32_ep25",
    "fullCon2_n45_linear_sess_char_unf1_lora1_lr0.001_els1_ctc1_bs32_ep100",
    "c1d_n45_conv1d_sess_char_unf1_lora1_lr0.001_els0.01_ctc1_bs32_ep30",
    "lrdif_n45_linear_sess_char_unf1_lora1_lr0.001_els0.01_ctc1_bs32_ep30",
    "deep_n45_deepd512b2s4_ic0.3_sess_char_unf1_lora1_lr0.001_els0.05_ctc1_bs32_ep150",
    "v2_n45_deepd512b2s2_ic0.3_wd0.01_sess_char_unf1_lora0_lr0.001_els0.05_ctc1_bs16_ep45",
    "v3_n45_deepd512b2s2_ic0.3_augn0.5o0.2_wd0.01_sess_char_unf1_lora0_lr0.001_els0.05_ctc1_bs16_ep60",
    "v4_n45_deepd512b2s2_ic0.3_augn0.8o0.2_wd0.01_sess_char_unf1_lora0_lr0.001_els0.05_ctc1_bs16_ep100",
    "v5_n45_deepd512b2s2_ic0.3_augn0.5o0.2_sm2_wd0.01_sess_char_unf1_lora0_lr0.001_els0.05_ctc1_bs16_ep60",
}
legacy_bpe_tags = {tag for tag in legacy_owsm_tags if "_char_" not in tag}

df["metadatos_inferidos"] = False
mask = df["exp_tag"].isin(legacy_owsm_tags) & df["init_encoder"].isna()
df.loc[mask, "init_encoder"] = "owsm"
df.loc[mask, "metadatos_inferidos"] = True

mask = df["exp_tag"].isin(legacy_bpe_tags) & df["objetivo_ctc"].isna()
df.loc[mask, "objetivo_ctc"] = "bpe"
df.loc[mask, "metadatos_inferidos"] = True

df["error"] = df["error"].fillna("").astype(str)
df["estado"] = np.select(
    [df["error"].eq(""), df["error"].str.contains("interrumpido", case=False, na=False)],
    ["completo", "interrumpido"],
    default="fallido",
)
df["es_ctc_puro"] = np.isclose(df["ctc_weight"], 1.0, equal_nan=False)
df["horas"] = df["minutos"] / 60.0


# El catálogo conserva el nombre físico histórico y añade el identificador científico.
catalog_path = RESULTS_DIR / "catalogo_ejecuciones.csv"
if catalog_path.exists():
    catalogo = pd.read_csv(catalog_path)
    catalog_cols = ["exp_tag_original", "id_principal", "experimento_principal", "usos"]
    df = df.merge(catalogo[catalog_cols], left_on="exp_tag", right_on="exp_tag_original", how="left")
    df["id_principal"] = df["id_principal"].fillna(df["nombre"])
else:
    catalogo = pd.DataFrame()
    df["id_principal"] = df["nombre"]
    df["experimento_principal"] = ""
    df["usos"] = ""

assert df["exp_tag"].is_unique, "exp_tag debe identificar de forma unica cada ejecucion"

print(f"{len(df)} ejecuciones cargadas")
display(df["estado"].value_counts().rename_axis("estado").to_frame("ejecuciones"))
display(pd.DataFrame({
    "campo": ["WER CTC", "CER textual", "WER beam OWSM", "curva por epoca"],
    "disponibles": [
        df["wer_ctc_greedy"].notna().sum(),
        df["cer_ctc_micro"].notna().sum(),
        df["wer_beam"].notna().sum(),
        sum((RESULTS_DIR / tag / "epocas.csv").exists() for tag in df["exp_tag"]),
    ],
}))


In [ ]:
RUNS = {
    "A01": "A1randEls_",
    "A02": "v3_",
    "A03": "A2congelado_",
    "A04": "v5_",
    "A05": "A1randLr1_",
    "A06": "A4randCong_",
    "C02": "H1semilla_",
    "C03": "H2semilla_",
    "C04": "H3owsmEls02_",
    "D01": "D0congExt_",
    "D02": "D1loraCong_",
    "D03": "D2ult2_",
    "D04": "D3ult4_",
    "D05": "D2bUlt2Lr1_",
    "D06": "D3bUlt4Lr1_",
    "E01": "C1linear_",
    "E02": "C2conv1d_",
    "F01": "F2L2_",
    "F02": "F1L3_",
    "G02": "B1fon_",
    "G03": "B2bpe256_",
    "H02": "E1hoIdent_",
    "H03": "E2hoMedia_",
    "H04": "E3hoSinSes_",
}

def obtener_run(clave_o_prefijo, required=True):
    prefijo = RUNS.get(clave_o_prefijo, clave_o_prefijo)
    candidatos = df[df["exp_tag"].str.startswith(prefijo, na=False)].copy()
    if candidatos.empty:
        if required:
            raise KeyError(f"No se encontro la ejecucion {clave_o_prefijo!r} ({prefijo})")
        return None
    candidatos = candidatos.sort_values(
        ["epocas_hechas", "wer_ctc_greedy"], ascending=[False, True], na_position="last"
    )
    return candidatos.iloc[0]

def mejor_nombre(nombre, metrica="wer_ctc_greedy"):
    candidatos = df[(df["nombre"] == nombre) & df[metrica].notna()].copy()
    if candidatos.empty:
        return None
    return candidatos.sort_values([metrica, "epocas_hechas"], ascending=[True, False]).iloc[0]

def tabla_runs(especificaciones):
    filas = []
    for clave, etiqueta in especificaciones:
        r = obtener_run(clave, required=False)
        if r is not None:
            item = r.to_dict()
            item["etiqueta"] = etiqueta
            filas.append(item)
    return pd.DataFrame(filas)

def cargar_epocas(clave_o_prefijo):
    r = obtener_run(clave_o_prefijo)
    ruta = RESULTS_DIR / r["exp_tag"] / "epocas.csv"
    if not ruta.exists():
        return pd.DataFrame()
    ep = pd.read_csv(ruta)
    for col in ep.columns:
        if col != "exp_tag":
            ep[col] = pd.to_numeric(ep[col], errors="coerce")
    return ep

def resumen_visible(tabla):
    cols = ["id_principal", "etiqueta", "objetivo_ctc", "init_encoder", "epocas_hechas",
            "cer_ctc_micro", "cer_ctc_greedy", "wer_ctc_greedy",
            "min_valid_cer_ctc", "compute_units"]
    return tabla[[c for c in cols if c in tabla]].copy()


## 2. A - Transferencia del encoder

La figura central utiliza un unico objetivo (caracter), la misma familia de adaptador y decodificacion CTC. Las curvas muestran el error CTC de validacion durante el entrenamiento; las barras incorporan tambien el WER textual final.

La conclusion valida queda restringida al regimen probado: **los pesos acusticos de OWSM no aportan ventaja bajo este esquema de adaptacion**, mientras que la misma arquitectura inicializada al azar si aprende la tarea.


In [ ]:
curvas_a = [
    ("A01", "A01 · Aleatorio, ajustado", COLORS["random"]),
    ("A02", "A02 · OWSM, ajustado", COLORS["owsm"]),
    ("A03", "A03 · OWSM, congelado", COLORS["light"]),
]

fig, ax = plt.subplots(figsize=(7.2, 4.6), constrained_layout=True)
datos_curvas = []
for clave, etiqueta, color in curvas_a:
    ep = cargar_epocas(clave)
    if ep.empty or "valid_cer_ctc" not in ep:
        continue
    serie = ep[["epoch", "valid_cer_ctc"]].dropna()
    ax.plot(serie["epoch"], serie["valid_cer_ctc"], label=etiqueta, color=color)
    j = serie["valid_cer_ctc"].idxmin()
    minimo = serie.loc[j]
    ax.scatter(minimo["epoch"], minimo["valid_cer_ctc"], color=color,
               edgecolor="white", linewidth=0.8, zorder=4)
    ax.annotate(f"{minimo['valid_cer_ctc']:.1%}\nep. {int(minimo['epoch'])}",
                (minimo["epoch"], minimo["valid_cer_ctc"]),
                xytext=(6, -4), textcoords="offset points", fontsize=8, color=color)
    aux = serie.copy()
    aux["ejecucion"] = clave
    datos_curvas.append(aux)

ax.set(title="Transferencia del encoder de OWSM", xlabel="Epoca",
       ylabel="CER CTC en validacion (menor es mejor)")
eje_porcentaje(ax, "y", 1.02)
ax.grid(axis="y", color="#DDDDDD", linewidth=0.8)
ax.legend(loc="upper right")
guardar(fig, "fig_01_A01_curvas_transferencia_encoder")
if datos_curvas:
    exportar_tabla(pd.concat(datos_curvas, ignore_index=True), "datos_01_A01_curvas_transferencia_encoder")
plt.show()


In [ ]:
bloque_a = tabla_runs([
    ("A01", "A01 · Aleatorio, 30 ep."),
    ("A02", "A02 · OWSM ajustado, 38 ep."),
    ("A04", "A04 · OWSM ajustado, 60 ep."),
    ("A03", "A03 · OWSM congelado, 30 ep."),
])

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.5), constrained_layout=True)
colores = [COLORS["random"] if x == "random" else COLORS["owsm"]
           for x in bloque_a["init_encoder"]]

for ax, metrica, titulo in [
    (axes[0], "wer_ctc_greedy", "Error de palabra final"),
    (axes[1], "cer_ctc_greedy", "Error de caracter final"),
]:
    orden = np.arange(len(bloque_a))
    valores = bloque_a[metrica].to_numpy(float)
    ax.barh(orden, valores, color=colores, alpha=0.9)
    ax.set_yticks(orden, bloque_a["etiqueta"])
    ax.invert_yaxis()
    ax.set_xlabel("WER" if metrica.startswith("wer") else "CER")
    ax.set_title(titulo)
    eje_porcentaje(ax)
    ax.grid(axis="x", color="#E0E0E0", linewidth=0.8)
    anotar_barras_h(ax, valores)

guardar(fig, "fig_01_A02_metricas_transferencia_encoder")
exportar_tabla(resumen_visible(bloque_a), "tabla_01_A_transferencia_encoder")
plt.show()
display(resumen_visible(bloque_a))

a1, v3 = obtener_run("A01"), obtener_run("A02")
delta_min = v3["min_valid_cer_ctc"] - a1["min_valid_cer_ctc"]
delta_wer = v3["wer_ctc_greedy"] - a1["wer_ctc_greedy"]
display(Markdown(
    f"**Lectura automatica.** Frente a v3, la inicializacion aleatoria reduce "
    f"el minimo de CER CTC en **{delta_min:.1%} puntos absolutos** y el WER textual "
    f"en **{delta_wer:.1%} puntos**, con menor presupuesto de entrenamiento."
))


## 3. C - Estabilidad del entrenamiento

Las semillas no sustituyen una evaluacion sobre participantes adicionales, pero permiten comprobar si el resultado aleatorio depende de una inicializacion afortunada. A01 tiene un presupuesto de 30 epocas y C02/C03 de 40, por lo que se muestran ambas metricas y se conserva esta diferencia en las etiquetas.


In [ ]:
semillas = tabla_runs([
    ("A01", "A01 · semilla 2024 (30 ep.)"),
    ("C02", "C02 · semilla 7 (40 ep.)"),
    ("C03", "C03 · semilla 1234 (40 ep.)"),
])

fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.2), constrained_layout=True)
for ax, metrica, titulo in [
    (axes[0], "min_valid_cer_ctc", "Minimo CER CTC"),
    (axes[1], "wer_ctc_greedy", "WER textual"),
]:
    valores = semillas[metrica].to_numpy(float)
    x = np.arange(len(semillas))
    ax.scatter(x, valores, s=70, color=COLORS["random"], zorder=3)
    media, desv = np.nanmean(valores), np.nanstd(valores, ddof=1)
    ax.axhline(media, color=COLORS["neutral"], linestyle="--", linewidth=1.5,
               label=f"media {media:.1%}; s={desv:.1%}")
    ax.set_xticks(x, semillas["etiqueta"], rotation=18, ha="right")
    ax.set_title(titulo)
    eje_porcentaje(ax, "y")
    ax.grid(axis="y", color="#E0E0E0", linewidth=0.8)
    ax.legend(loc="best")

guardar(fig, "fig_02_C01_estabilidad_semillas")
exportar_tabla(resumen_visible(semillas), "tabla_02_C_estabilidad_entrenamiento")
plt.show()


## 4. D - Estrategia de adaptacion

Este bloque ordena las estrategias de adaptacion del encoder OWSM. La linea discontinua es la referencia aleatoria. Los presupuestos no son identicos: D01 usa 70 epocas y A02 termina en 38; la tabla exportada conserva ese dato para evitar interpretar la linea como una curva asintotica exacta.


In [ ]:
conservacion = tabla_runs([
    ("D01", "D01 · congelado (70 ep.)"),
    ("D02", "D02 · LoRA (30 ep.)"),
    ("D03", "D03 · 2 capas (30 ep.)"),
    ("D04", "D04 · 4 capas (30 ep.)"),
    ("A02", "A02 · ajuste completo (38 ep.)"),
])
ref_random = obtener_run("A01")
x = np.arange(len(conservacion))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
for ax, metrica, referencia, titulo in [
    (axes[0], "wer_ctc_greedy", ref_random["wer_ctc_greedy"], "WER textual"),
    (axes[1], "cer_ctc_greedy", ref_random["cer_ctc_greedy"], "CER textual"),
]:
    valores = conservacion[metrica].to_numpy(float)
    ax.plot(x, valores, marker="o", color=COLORS["owsm"])
    ax.axhline(referencia, color=COLORS["random"], linestyle="--",
               label=f"Encoder aleatorio: {referencia:.1%}")
    if metrica == "cer_ctc_greedy" and "cer_ctc_greedy_se" in conservacion:
        err = 1.96 * conservacion["cer_ctc_greedy_se"].to_numpy(float)
        ax.errorbar(x, valores, yerr=err, fmt="none", ecolor=COLORS["owsm"], capsize=3)
    ax.set_xticks(x, conservacion["etiqueta"], rotation=22, ha="right")
    ax.set_title(titulo)
    eje_porcentaje(ax, "y")
    ax.grid(axis="y", color="#E0E0E0", linewidth=0.8)
    ax.legend(loc="best")

guardar(fig, "fig_03_D01_estrategia_adaptacion")
exportar_tabla(resumen_visible(conservacion), "tabla_03_D_estrategia_adaptacion")
plt.show()


## 5. E - Frontend de EEG

La comparacion emplea inicializacion aleatoria, objetivo de caracteres, capa de sesion, submuestreo por dos e InterCTC. Se presentan conjuntamente WER y coste para que una mejora pequena no oculte su precio computacional.


In [ ]:
frontends = tabla_runs([
    ("E01", "E01 · Lineal"),
    ("E02", "E02 · Convolucional 1D"),
    ("A01", "A01 · Profundo (referencia)"),
])

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), constrained_layout=True)
y = np.arange(len(frontends))
colores_f = [COLORS["light"], COLORS["green"], COLORS["random"]]

axes[0].barh(y, frontends["wer_ctc_greedy"], color=colores_f)
axes[0].set_yticks(y, frontends["etiqueta"])
axes[0].invert_yaxis()
axes[0].set(title="Calidad", xlabel="WER")
eje_porcentaje(axes[0])
axes[0].grid(axis="x", color="#E0E0E0", linewidth=0.8)
anotar_barras_h(axes[0], frontends["wer_ctc_greedy"].to_numpy(float))

axes[1].barh(y, frontends["compute_units"], color=colores_f)
axes[1].set_yticks(y, frontends["etiqueta"])
axes[1].invert_yaxis()
axes[1].set(title="Coste", xlabel="Unidades de computo")
axes[1].grid(axis="x", color="#E0E0E0", linewidth=0.8)
for patch, valor in zip(axes[1].patches, frontends["compute_units"]):
    axes[1].text(valor + 0.12, patch.get_y() + patch.get_height()/2, f"{valor:.1f}", va="center")

guardar(fig, "fig_04_E01_frontend_eeg")
exportar_tabla(resumen_visible(frontends), "tabla_04_E_frontend_eeg")
plt.show()


## 6. F - Profundidad del encoder

Tras comprobar que los pesos no ayudan, se estudia si la profundidad de E-Branchformer sigue siendo necesaria. Todos los puntos usan inicializacion aleatoria; las diferencias pequenas deben leerse junto con la variabilidad entre semillas.


In [ ]:
capas = tabla_runs([
    ("F01", "F01 · 2 bloques"),
    ("F02", "F02 · 3 bloques"),
    ("A01", "A01 · 6 bloques (referencia)"),
])
capas["bloques"] = [2, 3, 6]

fig, ax1 = plt.subplots(figsize=(7.2, 4.5), constrained_layout=True)
ax1.plot(capas["bloques"], capas["wer_ctc_greedy"], marker="o",
         color=COLORS["random"], label="WER")
ax1.set(xlabel="Numero de bloques E-Branchformer", ylabel="WER",
        title="Profundidad del encoder: calidad y coste")
ax1.set_xticks(capas["bloques"])
eje_porcentaje(ax1, "y")
ax1.grid(axis="y", color="#E0E0E0", linewidth=0.8)

ax2 = ax1.twinx()
ax2.plot(capas["bloques"], capas["compute_units"], marker="s",
         color=COLORS["green"], linestyle="--", label="Coste")
ax2.set_ylabel("Unidades de computo")
ax2.spines["right"].set_visible(True)

lineas = ax1.get_lines() + ax2.get_lines()
ax1.legend(lineas, [l.get_label() for l in lineas], loc="best")
guardar(fig, "fig_05_F01_profundidad_encoder")
exportar_tabla(resumen_visible(capas), "tabla_05_F_profundidad_encoder")
plt.show()


## 7. G - Unidad de salida CTC

El error nativo de entrenamiento no se compara entre objetivos. El fonema queda fuera del ranking textual porque no existe un conversor fonema-a-texto. Las barras usan WER despues de reconstruir texto; el panel derecho muestra la densidad de supervision disponible para cada clase.


In [ ]:
objetivos_stats_path = RESULTS_DIR / "objetivos_estadisticas.csv"
if objetivos_stats_path.exists():
    stats_obj = pd.read_csv(objetivos_stats_path)
else:
    stats_obj = pd.DataFrame({
        "objetivo": ["char", "fon", "bpe256", "word", "bpe-OWSM"],
        "clases": [29, 42, 257, 3731, 50002],
        "tokens_train": [np.nan] * 5,
        "ejemplos_por_clase": [np.nan] * 5,
    })

seleccion_obj = []
for objetivo, etiqueta, run in [
    ("char", "G01 · Caracter", obtener_run("A01")),
    ("fon", "G02 · Fonema", obtener_run("G02")),
    ("bpe256", "G03 · BPE propio (256)", obtener_run("G03")),
    ("word", "G04 · Palabra", mejor_nombre("B3word")),
    ("bpe-OWSM", "G05 · BPE OWSM (50k)", mejor_nombre("B4bpe50k")),
]:
    item = run.to_dict()
    item["objetivo_stats"] = objetivo
    item["etiqueta"] = etiqueta
    seleccion_obj.append(item)

objetivos = pd.DataFrame(seleccion_obj).merge(
    stats_obj, left_on="objetivo_stats", right_on="objetivo", how="left",
    suffixes=("", "_stats"),
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), constrained_layout=True)
con_wer = objetivos[objetivos["wer_ctc_greedy"].notna()].copy()
orden = np.arange(len(con_wer))
colores_o = [COLORS["random"], COLORS["green"], COLORS["yellow"], COLORS["owsm"]]
axes[0].barh(orden, con_wer["wer_ctc_greedy"], color=colores_o[:len(con_wer)])
axes[0].set_yticks(orden, con_wer["etiqueta"])
axes[0].invert_yaxis()
axes[0].set(title="Rendimiento del texto reconstruido", xlabel="WER")
eje_porcentaje(axes[0])
axes[0].grid(axis="x", color="#E0E0E0", linewidth=0.8)
anotar_barras_h(axes[0], con_wer["wer_ctc_greedy"].to_numpy(float))

validos = objetivos[objetivos["ejemplos_por_clase"].notna()].copy()
axes[1].scatter(validos["clases"], validos["ejemplos_por_clase"],
                s=75, color=[COLORS["random"], COLORS["purple"], COLORS["green"],
                             COLORS["yellow"], COLORS["owsm"]][:len(validos)])
for _, r in validos.iterrows():
    axes[1].annotate(r["etiqueta"], (r["clases"], r["ejemplos_por_clase"]),
                     xytext=(5, 4), textcoords="offset points", fontsize=8)
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set(title="Densidad de supervision", xlabel="Numero de clases",
            ylabel="Tokens de entrenamiento por clase")
axes[1].grid(color="#E0E0E0", linewidth=0.7, which="both")

guardar(fig, "fig_06_G01_unidades_salida_ctc")
cols_obj = ["etiqueta", "clases", "tokens_train", "ejemplos_por_clase",
            "epocas_hechas", "cer_ctc_micro", "wer_ctc_greedy", "compute_units"]
exportar_tabla(objetivos[cols_obj], "tabla_06_G_unidades_salida_ctc")
plt.show()
display(objetivos[cols_obj])

mejor_obj = con_wer.loc[con_wer["wer_ctc_greedy"].idxmin()]
display(Markdown(
    f"**Lectura automatica.** El menor WER disponible corresponde a **{mejor_obj['etiqueta']}** "
    f"con **{mejor_obj['wer_ctc_greedy']:.1%}**. El objetivo fonetico no entra en esta "
    "comparacion porque no se implemento el paso de fonemas a ortografia."
))


## 8. H - Generalizacion entre sesiones

Las tres variantes se entrenan excluyendo cinco sesiones. La referencia A01 utiliza sesiones vistas; sirve para cuantificar la magnitud del cambio de dominio, no como una comparacion perfectamente emparejada.


In [ ]:
holdout = tabla_runs([
    ("A01", "A01 · sesiones vistas (referencia)"),
    ("H02", "H02 · no vista: identidad"),
    ("H03", "H03 · no vista: matriz media"),
    ("H04", "H04 · no vista: sin capa de sesion"),
])

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.5), constrained_layout=True)
colores_h = [COLORS["random"], COLORS["owsm"], COLORS["purple"], COLORS["green"]]
y = np.arange(len(holdout))
for ax, metrica, titulo in [
    (axes[0], "wer_ctc_greedy", "Error de palabra"),
    (axes[1], "cer_ctc_greedy", "Error de caracter"),
]:
    valores = holdout[metrica].to_numpy(float)
    ax.barh(y, valores, color=colores_h)
    ax.set_yticks(y, holdout["etiqueta"])
    ax.invert_yaxis()
    ax.set_title(titulo)
    ax.set_xlabel("WER" if metrica.startswith("wer") else "CER")
    eje_porcentaje(ax)
    ax.grid(axis="x", color="#E0E0E0", linewidth=0.8)
    anotar_barras_h(ax, valores)

guardar(fig, "fig_07_H01_generalizacion_sesiones")
exportar_tabla(resumen_visible(holdout), "tabla_07_H_generalizacion_sesiones")
plt.show()


## 9. I - Modelo de lenguaje externo

`pyctcdecode` y KenLM operan sobre la rama CTC: no son el decoder Transformer de OWSM. Se compara cada ejecucion consigo misma para medir la reduccion de WER debida al modelo de lenguaje. El `oracle` representa la mejor hipotesis disponible en la lista acustica y no es un sistema desplegable.


In [ ]:
lm_frames = []
for ruta in sorted(RESULTS_DIR.glob("*/lm_resultados.csv")):
    aux = pd.read_csv(ruta)
    aux["carpeta"] = ruta.parent.name
    lm_frames.append(aux)

lm = pd.concat(lm_frames, ignore_index=True) if lm_frames else pd.DataFrame()
if not lm.empty:
    nombres_lm = {
        "A1randEls": "I01 · A01",
        "D0congExt": "I02 · D01",
        "E1hoIdent": "I03 · H02",
        "F1L3": "I04 · F02",
        "H1semilla": "I05 · C02",
        "v4": "I06 · P (v4)",
        "v5": "I07 · A04",
    }
    lm["modelo"] = lm["exp_tag"].str.split("_").str[0].map(nombres_lm).fillna(
        lm["exp_tag"].str.split("_").str[0]
    )
    orden_metodos = ["greedy", "2-grama", "3-grama", "oraculo"]
    piv = lm.pivot_table(index="modelo", columns="metodo", values="wer", aggfunc="first")
    piv = piv[[c for c in orden_metodos if c in piv.columns]].sort_index()

    fig, ax = plt.subplots(figsize=(8.5, 4.8), constrained_layout=True)
    x = np.arange(len(piv))
    estilos = {
        "greedy": (COLORS["neutral"], "o", "-"),
        "2-grama": (COLORS["green"], "s", "-"),
        "3-grama": (COLORS["owsm"], "^", "-"),
        "oraculo": (COLORS["light"], "D", "--"),
    }
    for metodo in piv.columns:
        color, marker, ls = estilos[metodo]
        ax.plot(x, piv[metodo], color=color, marker=marker, linestyle=ls, label=metodo)
    ax.set_xticks(x, piv.index)
    ax.set(title="Efecto del modelo de lenguaje sobre cada encoder",
           xlabel="Ejecucion", ylabel="WER")
    eje_porcentaje(ax, "y")
    ax.grid(axis="y", color="#E0E0E0", linewidth=0.8)
    ax.legend(ncol=2)
    guardar(fig, "fig_08_I01_modelo_lenguaje_wer")
    exportar_tabla(lm, "datos_08_I01_modelo_lenguaje")
    plt.show()

    if "greedy" in piv and any(c in piv for c in ("2-grama", "3-grama")):
        lm_cols = [c for c in ("2-grama", "3-grama") if c in piv]
        mejora = piv["greedy"] - piv[lm_cols].min(axis=1)
        display(Markdown(
            f"**Lectura automatica.** La mejor fusion reduce el WER entre "
            f"**{mejora.min():.1%} y {mejora.max():.1%} puntos absolutos** "
            "en las ejecuciones con resultados completos."
        ))
else:
    display(Markdown("No se encontraron archivos `lm_resultados.csv`."))


In [ ]:
rejillas = sorted(RESULTS_DIR.glob("*/lm_rejilla.csv"))
if rejillas:
    rejilla = pd.read_csv(rejillas[0]).dropna(subset=["alpha", "beta", "wer"])
    mapa = rejilla.pivot_table(index="alpha", columns="beta", values="wer", aggfunc="first")
    mapa = mapa.sort_index().sort_index(axis=1)

    fig, ax = plt.subplots(figsize=(6.2, 4.5), constrained_layout=True)
    im = ax.imshow(mapa.to_numpy(), cmap="YlGnBu_r", aspect="auto")
    ax.set_xticks(range(len(mapa.columns)), [f"{x:g}" for x in mapa.columns])
    ax.set_yticks(range(len(mapa.index)), [f"{x:g}" for x in mapa.index])
    ax.set(xlabel="beta (insercion de palabras)", ylabel="alpha (peso del LM)",
           title="Sensibilidad del WER a la fusion superficial")
    for i in range(len(mapa.index)):
        for j in range(len(mapa.columns)):
            valor = mapa.iloc[i, j]
            ax.text(j, i, f"{valor:.1%}", ha="center", va="center", fontsize=9,
                    color="white" if valor > np.nanmedian(mapa.to_numpy()) else "black")
    cb = fig.colorbar(im, ax=ax, shrink=0.85)
    cb.ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    cb.set_label("WER")
    guardar(fig, "fig_08_I02_modelo_lenguaje_rejilla")
    exportar_tabla(rejilla, "datos_08_I01_modelo_lenguaje_rejilla")
    plt.show()


## 10. J - Decoder OWSM preliminar

Estas ejecuciones son anteriores a la receta final. Con `ctc_weight=0.3`, el decoder de atencion participa en la perdida y en el beam search. No constituyen una ablacion controlada del decoder, pero documentan por que se paso a CTC puro: el CER con *teacher forcing* parece moderado mientras la generacion libre mantiene un WER alrededor o por encima del 100 %.


In [ ]:
decoder = df[(df["ctc_weight"] < 1.0) & df["cer_beam"].notna()].copy()
decoder = decoder.sort_values("wer_beam")
decoder["etiqueta"] = decoder["id_principal"]

if not decoder.empty:
    fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.5), constrained_layout=True)
    y = np.arange(len(decoder))
    h = 0.36
    axes[0].barh(y - h/2, decoder["best_valid_cer"], height=h,
                 color=COLORS["light"], label="CER con teacher forcing")
    axes[0].barh(y + h/2, decoder["cer_beam"], height=h,
                 color=COLORS["owsm"], label="CER generacion libre")
    axes[0].set_yticks(y, decoder["etiqueta"])
    axes[0].invert_yaxis()
    axes[0].set(title="Teacher forcing frente a inferencia", xlabel="CER")
    eje_porcentaje(axes[0])
    axes[0].grid(axis="x", color="#E0E0E0", linewidth=0.8)
    axes[0].legend()

    axes[1].barh(y, decoder["wer_beam"], color=COLORS["red"])
    axes[1].axvline(1.0, color=COLORS["neutral"], linestyle="--", linewidth=1.4,
                    label="WER = 100 %")
    axes[1].set_yticks(y, decoder["etiqueta"])
    axes[1].invert_yaxis()
    axes[1].set(title="Decoder OWSM en generacion libre", xlabel="WER beam")
    eje_porcentaje(axes[1])
    axes[1].grid(axis="x", color="#E0E0E0", linewidth=0.8)
    anotar_barras_h(axes[1], decoder["wer_beam"].to_numpy(float), offset=0.015)
    axes[1].legend()

    guardar(fig, "fig_09_J01_decoder_owsm_preliminar")
    exportar_tabla(resumen_visible(decoder), "tabla_09_J_decoder_owsm_preliminar")
    plt.show()


## 11. B - Sistema final: coste y frontera de Pareto

El bloque B queda deliberadamente al final y permanece abierto mientras termina el entrenamiento campeón. La figura de coste es descriptiva: mezcla decisiones experimentales y sirve para identificar configuraciones dominadas, no para sustituir las ablaciones anteriores.


In [ ]:
coste = df[
    df["es_ctc_puro"]
    & df["wer_ctc_greedy"].notna()
    & df["compute_units"].notna()
    & (df["n_train"] >= 7000)
    & (df["estado"] == "completo")
    & (df["holdout_sesiones"].fillna(0) == 0)
].copy()

coste = coste.sort_values("compute_units")
mejor_hasta = np.inf
pareto_idx = []
for idx, r in coste.iterrows():
    if r["wer_ctc_greedy"] < mejor_hasta:
        pareto_idx.append(idx)
        mejor_hasta = r["wer_ctc_greedy"]
coste["pareto"] = coste.index.isin(pareto_idx)

fig, ax = plt.subplots(figsize=(8.2, 5.1), constrained_layout=True)
for init, grupo in coste.groupby("init_encoder", dropna=False):
    color = COLORS["random"] if init == "random" else COLORS["owsm"]
    label = "Encoder aleatorio" if init == "random" else "Encoder OWSM"
    ax.scatter(grupo["compute_units"], grupo["wer_ctc_greedy"], s=48,
               alpha=0.72, color=color, label=label)

frente = coste[coste["pareto"]].sort_values("compute_units")
ax.plot(frente["compute_units"], frente["wer_ctc_greedy"], color=COLORS["neutral"],
        linestyle="--", linewidth=1.4, label="Frontera observada")

destacados = pd.concat([
    coste.nsmallest(5, "wer_ctc_greedy"),
    frente,
]).drop_duplicates("exp_tag")
for _, r in destacados.iterrows():
    etiqueta = f"{r['id_principal']} ({int(r['epocas_hechas'])} ep.)"
    ax.annotate(etiqueta, (r["compute_units"], r["wer_ctc_greedy"]),
                xytext=(5, 4), textcoords="offset points", fontsize=7.5)

ax.set(title="Compromiso entre coste computacional y error de palabra",
       xlabel="Unidades de computo", ylabel="WER")
eje_porcentaje(ax, "y")
ax.grid(color="#E0E0E0", linewidth=0.8)
ax.legend()
guardar(fig, "fig_10_B02_coste_frente_wer")
exportar_tabla(coste, "datos_10_B02_coste_frente_wer")
plt.show()


### B.1. Ranking provisional por WER

El ranking se restringe a CTC puro, entrenamientos completos con al menos 7.000 ensayos y evaluacion fuera del bloque de sesiones no vistas. Hasta incorporar B01, identifica el candidato actual, no un resultado final cerrado.


In [ ]:
ranking = df[
    df["es_ctc_puro"]
    & df["wer_ctc_greedy"].notna()
    & (df["n_train"] >= 7000)
    & (df["estado"] == "completo")
    & (df["holdout_sesiones"].fillna(0) == 0)
].copy().nsmallest(12, "wer_ctc_greedy")

ranking["etiqueta"] = ranking.apply(
    lambda r: f"{r['id_principal']} · {r['objetivo_ctc']} · {int(r['epocas_hechas'])} ep.", axis=1
)
ranking = ranking.sort_values("wer_ctc_greedy", ascending=False)

fig, ax = plt.subplots(figsize=(8.5, 5.8), constrained_layout=True)
colores_r = [COLORS["random"] if x == "random" else COLORS["owsm"]
             for x in ranking["init_encoder"]]
ax.barh(np.arange(len(ranking)), ranking["wer_ctc_greedy"], color=colores_r)
ax.set_yticks(np.arange(len(ranking)), ranking["etiqueta"])
ax.set(title="Mejores ejecuciones por error de palabra", xlabel="WER")
eje_porcentaje(ax)
ax.grid(axis="x", color="#E0E0E0", linewidth=0.8)
anotar_barras_h(ax, ranking["wer_ctc_greedy"].to_numpy(float))
guardar(fig, "fig_10_B01_ranking_sistema_final")

cols_rank = ["id_principal", "nombre", "exp_tag", "objetivo_ctc", "init_encoder", "frontend",
             "epocas_hechas", "cer_ctc_micro", "wer_ctc_greedy",
             "minutos", "compute_units"]
exportar_tabla(ranking[cols_rank].sort_values("wer_ctc_greedy"), "tabla_10_B_sistema_final")
plt.show()
display(ranking[cols_rank].sort_values("wer_ctc_greedy"))


## 12. Sintesis automatica para la memoria

Estas frases se generan desde los CSV actuales. Son un borrador factual: antes de incorporarlas literalmente a la memoria deben mantenerse sus condiciones de validez, especialmente que todas las cifras proceden de validacion de un unico participante.


In [ ]:
lineas = []

a01, a02, c02, c03 = (obtener_run(k) for k in ("A01", "A02", "C02", "C03"))
lineas.append(
    f"- **Transferencia:** A01 (aleatorio) alcanza {a01['wer_ctc_greedy']:.1%} WER "
    f"frente a {a02['wer_ctc_greedy']:.1%} de A02 (OWSM), una diferencia de "
    f"{a02['wer_ctc_greedy'] - a01['wer_ctc_greedy']:.1%} puntos a favor del aleatorio."
)

wers_semilla = np.array([a01["wer_ctc_greedy"], c02["wer_ctc_greedy"], c03["wer_ctc_greedy"]])
lineas.append(
    f"- **Reproducibilidad:** las tres semillas aleatorias obtienen un WER medio de "
    f"{wers_semilla.mean():.1%} con desviacion estandar de {wers_semilla.std(ddof=1):.1%}."
)

mejor = df[
    df["es_ctc_puro"] & df["wer_ctc_greedy"].notna()
    & (df["n_train"] >= 7000) & (df["estado"] == "completo")
    & (df["holdout_sesiones"].fillna(0) == 0)
].sort_values("wer_ctc_greedy").iloc[0]
lineas.append(
    f"- **Candidato actual del bloque B:** {mejor['id_principal']} con objetivo {mejor['objetivo_ctc']} "
    f"alcanza {mejor['wer_ctc_greedy']:.1%} WER y {mejor['cer_ctc_micro']:.1%} CER micro."
)

h02, h03, h04 = (obtener_run(k) for k in ("H02", "H03", "H04"))
mejor_holdout = min([h02, h03, h04], key=lambda r: r["wer_ctc_greedy"])
lineas.append(
    f"- **Sesiones no vistas:** la mejor variante es {mejor_holdout['nombre']} con "
    f"{mejor_holdout['wer_ctc_greedy']:.1%} WER, frente a {a01['wer_ctc_greedy']:.1%} "
    "en la referencia con sesiones vistas."
)

if not lm.empty:
    mejor_lm = lm[lm["metodo"].isin(["2-grama", "3-grama"])].sort_values("wer").iloc[0]
    lineas.append(
        f"- **Modelo de lenguaje:** el mejor resultado almacenado es "
        f"{mejor_lm['wer']:.1%} WER ({mejor_lm['metodo']}, {mejor_lm['modelo']}); "
        "KenLM mejora la busqueda CTC, pero no es el decoder de OWSM."
    )

if not decoder.empty:
    mejor_dec = decoder.sort_values("wer_beam").iloc[0]
    lineas.append(
        f"- **Decoder OWSM preliminar:** su mejor ejecucion libre obtiene "
        f"{mejor_dec['wer_beam']:.1%} WER; estos ensayos justifican el diagnostico "
        "CTC posterior, pero no constituyen una ablacion definitiva del decoder."
    )

display(Markdown("\n".join(lineas)))


## 13. Limitaciones que deben acompanar a las figuras

- Todas las cifras corresponden a validacion del participante T15; no existe evaluacion etiquetada sobre la prueba oficial.
- Los bloques con presupuestos diferentes no prueban igualdad de rendimiento asintotico.
- El WER de fonemas no esta disponible porque falta un decodificador fonema-lexico-texto.
- Los resultados con KenLM no miden el decoder Transformer de OWSM.
- Las cinco pruebas iniciales con decoder OWSM son preliminares y no aislan inicializacion del decoder, congelacion y peso de atencion.
- El mejor resultado con encoder aleatorio demuestra capacidad de la arquitectura, no transferencia de conocimiento desde audio.


In [ ]:
manifest = []
for ruta in sorted(FIG_DIR.glob("fig_*.png")):
    manifest.append({"tipo": "figura", "archivo": ruta.name, "ruta": str(ruta)})
for ruta in sorted(TABLE_DIR.glob("tabla_*.csv")):
    manifest.append({"tipo": "tabla", "archivo": ruta.name, "ruta": str(ruta)})

manifest = pd.DataFrame(manifest)
manifest.to_csv(TABLE_DIR / "manifest_artefactos.csv", index=False)
print(f"Generados {sum(manifest['tipo'].eq('figura'))} graficos PNG (+ PDF)")
print(f"Generadas {sum(manifest['tipo'].eq('tabla'))} tablas CSV (+ LaTeX)")
display(manifest)
